# Gradio demo: Text-to-speech (simple)

In [ ]:
from functools import lru_cache

import gradio as gr
import numpy as np
import torch


DEFAULT_TEXT = (
    "This is a local text to speech demo. Kokoro has named voices; "
    "MMS/VITS has one checkpoint per language."
)

KOKORO_VOICES = {
    "English (US)": {
        "af_heart - female": ("a", "af_heart"),
        "af_bella - female": ("a", "af_bella"),
        "af_nicole - female": ("a", "af_nicole"),
        "af_sarah - female": ("a", "af_sarah"),
        "am_adam - male": ("a", "am_adam"),
        "am_fenrir - male": ("a", "am_fenrir"),
        "am_michael - male": ("a", "am_michael"),
    },
    "English (UK)": {
        "bf_emma - female": ("b", "bf_emma"),
        "bf_isabella - female": ("b", "bf_isabella"),
        "bm_daniel - male": ("b", "bm_daniel"),
        "bm_george - male": ("b", "bm_george"),
    },
}

MMS_MODELS = {
    "English": "facebook/mms-tts-eng",
    "Spanish": "facebook/mms-tts-spa",
    "French": "facebook/mms-tts-fra",
    "German": "facebook/mms-tts-deu",
    "Hindi": "facebook/mms-tts-hin",
    "Japanese": "facebook/mms-tts-jpn",
}


def device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def audio_tuple(rate, audio):
    return rate, np.asarray(audio, dtype=np.float32).squeeze()


def require_text(text):
    text = text.strip()
    if not text:
        raise gr.Error("Enter some text first.")
    return text


@lru_cache
def kokoro_pipeline(lang_code):
    from kokoro import KPipeline

    return KPipeline(lang_code=lang_code, repo_id="hexgrad/Kokoro-82M")


def kokoro_voices_for(language_label):
    return list(KOKORO_VOICES[language_label])


def update_kokoro_language(language_label):
    voices = kokoro_voices_for(language_label)
    return gr.update(choices=voices, value=voices[0])


def run_kokoro(text, language_label, voice_label, speed):
    text = require_text(text)
    lang_code, voice = KOKORO_VOICES[language_label][voice_label]
    pieces = kokoro_pipeline(lang_code)(text, voice=voice, speed=speed)
    audio = np.concatenate([piece for _, _, piece in pieces])
    return audio_tuple(24000, audio), f"Kokoro voice: {voice}"


@lru_cache
def mms_model(model_id):
    from transformers import AutoTokenizer, VitsModel

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = VitsModel.from_pretrained(model_id).to(device())
    return tokenizer, model


def run_mms(text, language_label, speaking_rate, noise_scale, seed):
    text = require_text(text)
    torch.manual_seed(int(seed))
    model_id = MMS_MODELS[language_label]
    tokenizer, model = mms_model(model_id)
    model.speaking_rate = speaking_rate
    model.noise_scale = noise_scale
    inputs = tokenizer(text, return_tensors="pt").to(device())

    with torch.inference_mode():
        audio = model(**inputs).waveform

    return audio_tuple(model.config.sampling_rate, audio.cpu().numpy()), model_id


with gr.Blocks(title="Modern local TTS demo") as demo:
    gr.Markdown("# Modern local TTS demo")

    text = gr.Textbox(label="Text", value=DEFAULT_TEXT, lines=5)
    output = gr.Audio(label="Output")
    notes = gr.Markdown()

    with gr.Tabs():
        with gr.Tab("Kokoro"):
            kokoro_language = gr.Dropdown(
                label="Language",
                choices=list(KOKORO_VOICES),
                value="English (US)",
            )
            kokoro_voice = gr.Dropdown(
                label="Voice",
                choices=kokoro_voices_for("English (US)"),
                value="af_heart - female",
            )
            kokoro_speed = gr.Slider(0.7, 1.3, value=1.0, step=0.05, label="Speed")
            kokoro_button = gr.Button("Generate with Kokoro", variant="primary")

        with gr.Tab("MMS / VITS"):
            mms_language = gr.Dropdown(
                label="Language checkpoint",
                choices=list(MMS_MODELS),
                value="English",
            )
            mms_rate = gr.Slider(0.6, 1.6, value=1.0, step=0.05, label="Speaking rate")
            mms_noise = gr.Slider(0.0, 1.2, value=0.667, step=0.05, label="Noise")
            mms_seed = gr.Number(label="Seed", value=0, precision=0)
            mms_button = gr.Button("Generate with MMS", variant="primary")

    kokoro_language.change(update_kokoro_language, kokoro_language, kokoro_voice)
    kokoro_button.click(run_kokoro, [text, kokoro_language, kokoro_voice, kokoro_speed], [output, notes])
    mms_button.click(run_mms, [text, mms_language, mms_rate, mms_noise, mms_seed], [output, notes])


# Does not need a guard
demo.launch()
